# Explainable Loan Default Risk Prediction Using Machine Learning
## 1.1 Executive Summary 

# 2.0 Business Understanding

## 2.1 Stakeholder

The Credit Risk Manager is responsible for assessing the creditworthiness of loan applicants and managing the company's lending risk. They make daily decisions regarding loan approvals, loan pricing, customer eligibility, and risk mitigation strategies to ensure sustainable lending operations.

This person needs to decide, for each loan application, whether to approve or reject it based on predicted risk of default.

---

## 2.2 Business Problem

How can a lending institution predict, before issuing a loan, which applicants are likely to default so that the institution can reduce financial losses while maintaining fair access to credit?

Without a prediction model, the institution must either reject all risky-looking applicants (losing potentially good customers) or approve too many applicants (accumulating bad debt and increasing financial losses).

---

## 2.3 Business Objectives

The primary objective is to reduce loan default rates by identifying high-risk borrowers before loan approval.

Specific objectives include:

* Improving the accuracy of credit risk assessment.
* Supporting faster and more consistent lending decisions.
* Reducing financial losses associated with customer defaults.
* Increasing access to credit for creditworthy customers.
* Enhancing the sustainability and profitability of lending operations.

---

## 2.4 Data Science Objectives

The data science objective is to develop and evaluate machine learning models capable of predicting whether a customer is likely to default on a loan.

Specific objectives include:

* Building predictive classification models using customer demographic, credit history, and repayment data.
* Comparing the performance of Logistic Regression, Random Forest, and XGBoost models.
* Identifying the most important drivers of loan default.
* Applying explainable AI techniques to improve transparency and stakeholder trust.
* Generating actionable insights to support credit risk management decisions.

---

## 2.5 Success Criteria

### Business Success Metrics

The model must identify at least 70% of actual defaulters. Missing a true defaulter can result in substantial financial losses due to unpaid loans.

The model should maintain a Precision of at least 0.60. Excessively flagging low-risk customers as risky may lead to lost business opportunities and reduced customer satisfaction.

The model should achieve an ROC-AUC score of at least 0.75, demonstrating meaningful predictive ability beyond random guessing.

These targets will be used later to evaluate whether the final model is suitable for supporting lending decisions.

---

## 2.6 Business Understanding

## 2.6.1 Stakeholder

The Credit Risk Manager at M-KOPA Kenya is responsible for assessing the creditworthiness of loan applicants and managing the company's lending risk. They make daily decisions regarding loan approvals, loan pricing, customer eligibility, and risk mitigation strategies to ensure sustainable lending operations.

This person needs to decide, for each loan application, whether to approve or reject it based on predicted risk of default.

---

## 2.6.2 Business Problem

How can a Kenyan lending institution predict, before issuing a loan, which applicants are likely to default so that the institution can reduce financial losses while maintaining fair access to credit?

Without a prediction model, the institution must either reject all risky-looking applicants (losing potentially good customers) or approve too many applicants (accumulating bad debt and increasing financial losses).

---

## 2.6.3 Business Objectives

The primary objective is to reduce loan default rates by identifying high-risk borrowers before loan approval.

Specific objectives include:

* Improving the accuracy of credit risk assessment.
* Supporting faster and more consistent lending decisions.
* Reducing financial losses associated with customer defaults.
* Increasing access to credit for creditworthy customers.
* Enhancing the sustainability and profitability of lending operations.

---

## 2.6.4 Data Science Objectives

The data science objective is to develop and evaluate machine learning models capable of predicting whether a customer is likely to default on a loan.

Specific objectives include:

* Building predictive classification models using customer demographic, credit history, and repayment data.
* Comparing the performance of Logistic Regression, Random Forest, and XGBoost models.
* Identifying the most important drivers of loan default.
* Applying explainable AI techniques to improve transparency and stakeholder trust.
* Generating actionable insights to support credit risk management decisions.

---

## 2.7 Success Criteria

### Business Success Metrics


The model must identify at least 70% of actual defaulters (Recall ≥ 0.70). Missing a true defaulter can result in substantial financial losses due to unpaid loans.

The model should maintain a Precision of at least 0.60. Excessively flagging low-risk customers as risky may lead to lost business opportunities and reduced customer satisfaction.

The model should achieve an ROC-AUC score of at least 0.75, demonstrating meaningful predictive ability beyond random guessing.

These targets will be used later to evaluate whether the final model is suitable for supporting lending decisions.

---

## 2.8 Kenyan Context and Project Relevance


Digital lending plays a significant role in Kenya's financial sector, with organizations such as M-KOPA, Tala, Branch, and major banks using data-driven credit assessment to serve millions of customers. As lending grows, accurate and explainable credit risk models are essential for reducing defaults while promoting financial inclusion.

Although this project uses data from Home Credit's Eastern European operations, the methodology is directly applicable to Kenyan digital lending. The techniques used to predict default risk and explain lending decisions are highly relevant to modern credit scoring systems in Kenya.



# 3.0 Data Understanding

## 3.1 Dataset Description

The Home Credit Default Risk dataset consists of multiple related tables containing customer information, credit history, previous applications, repayment behavior, and credit card activity. These datasets will be combined to create a customer-level dataset for predicting loan default risk.

### application_train.csv

The primary dataset containing one record per loan application. It includes customer demographic and financial information as well as the target variable (`TARGET`), where 1 indicates default and 0 indicates successful repayment.

**Join Key:** `SK_ID_CURR`

### bureau.csv

Contains the applicant's credit history from other financial institutions reported to the credit bureau. Since multiple records may exist per customer, aggregation is required before merging.

**Join Key:** `SK_ID_CURR`

### previous_application.csv

Contains records of all previous loan applications submitted to Home Credit. It provides insights into past borrowing behavior and lending decisions.

**Join Keys:** `SK_ID_CURR`, `SK_ID_PREV`

### installments_payments.csv

Contains repayment history for previous Home Credit loans, including payment timing and amounts. This dataset helps assess repayment discipline and must be aggregated before merging.

**Join Key:** `SK_ID_PREV`

### credit_card_balance.csv

Contains monthly credit card balance snapshots, including credit utilization and outstanding balances. It provides insights into how customers manage revolving credit.

**Join Key:** `SK_ID_CURR`

### Relational Structure

The analysis will use `application_train.csv` as the master dataset. Additional customer-level features will be created by aggregating and merging information from the other datasets using `SK_ID_CURR` and `SK_ID_PREV`.


## 4.0 Data Loading
#### Loading the Home Credit CSV files in a portable, and repeatable. This ensures that file paths are managed centrally, making the notebook more organized, reusable, and portable across different environments.

In [6]:
# Setting DATA_DIR and file path helper
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# Update this path to the folder that contains your Home Credit CSV files.
DATA_DIR = Path(r"C:\Users\Kim\Desktop\Loan Default Predictions")


def data_path(filename: str) -> Path:
    """Build a portable path to a file in the data directory."""
    return DATA_DIR / filename


DATA_DIR

WindowsPath('C:/Users/Kim/Desktop/Loan Default Predictions')

In [7]:
# Loading each file with pd.read_csv()
def print_load_check(name: str, df: pd.DataFrame, target_column: str | None = None) -> None:
    print(f"{name}: shape = {df.shape}")
    display(df.head(3))

    if target_column is not None:
        print(f"{name}: {target_column} distribution")
        display(
            df[target_column]
            .value_counts(normalize=True)
            .rename("proportion")
            .mul(100)
            .round(2)
            .to_frame()
        )


application_train = pd.read_csv(data_path("application_train.csv"))
print_load_check("application_train", application_train, target_column="TARGET")

bureau = pd.read_csv(data_path("bureau.csv"))
print_load_check("bureau", bureau)

previous_application = pd.read_csv(data_path("previous_application.csv"))
print_load_check("previous_application", previous_application)

installments_payments = pd.read_csv(data_path("installments_payments.csv"))
print_load_check("installments_payments", installments_payments)

credit_card_balance = pd.read_csv(data_path("credit_card_balance.csv"))
print_load_check("credit_card_balance", credit_card_balance)

application_train: shape = (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


application_train: TARGET distribution


,proportion
TARGET,
0,91.93
1,8.07


bureau: shape = (1716428, 17)


...

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0


installments_payments: shape = (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000


credit_card_balance: shape = (3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0


In [8]:
# Summaries of each file
print("application_train summary:")
display(application_train.describe(include="all").T)
print("bureau summary:")
display(bureau.describe(include="all").T)
print("previous_application summary:")
display(previous_application.describe(include="all").T)
print("installments_payments summary:")
display(installments_payments.describe(include="all").T)
print("credit_card_balance summary:")
display(credit_card_balance.describe(include="all").T)

application_train summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,NaN,NaN,NaN,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
TARGET,307511.0,NaN,NaN,NaN,0.080729,0.272419,0.0,0.0,0.0,0.0,1.0
NAME_CONTRACT_TYPE,307511,2,Cash loans,278232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CODE_GENDER,307511,3,F,202448,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FLAG_OWN_CAR,307511,2,N,202924,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,265992.0,NaN,NaN,NaN,0.007,0.110757,0.0,0.0,0.0,0.0,9.0
AMT_REQ_CREDIT_BUREAU_WEEK,265992.0,NaN,NaN,NaN,0.034362,0.204685,0.0,0.0,0.0,0.0,8.0
AMT_REQ_CREDIT_BUREAU_MON,265992.0,NaN,NaN,NaN,0.267395,0.916002,0.0,0.0,0.0,0.0,27.0
AMT_REQ_CREDIT_BUREAU_QRT,265992.0,NaN,NaN,NaN,0.265474,0.794056,0.0,0.0,0.0,0.0,261.0


bureau summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_CURR,1716428.0,NaN,NaN,NaN,278214.933645,102938.558112,100001.0,188866.75,278055.0,367426.0,456255.0
SK_ID_BUREAU,1716428.0,NaN,NaN,NaN,5924434.489032,532265.728552,5000000.0,5463953.75,5926303.5,6385681.25,6843457.0
CREDIT_ACTIVE,1716428,4,Closed,1079273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CREDIT_CURRENCY,1716428,4,currency 1,1715020,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DAYS_CREDIT,1716428.0,NaN,NaN,NaN,-1142.107685,795.164928,-2922.0,-1666.0,-987.0,-474.0,0.0
CREDIT_DAY_OVERDUE,1716428.0,NaN,NaN,NaN,0.818167,36.544428,0.0,0.0,0.0,0.0,2792.0
DAYS_CREDIT_ENDDATE,1610875.0,NaN,NaN,NaN,510.517362,4994.219837,-42060.0,-1138.0,-330.0,474.0,31199.0
DAYS_ENDDATE_FACT,1082775.0,NaN,NaN,NaN,-1017.437148,714.010626,-42023.0,-1489.0,-897.0,-425.0,0.0
AMT_CREDIT_MAX_OVERDUE,591940.0,NaN,NaN,NaN,3825.417661,206031.606207,0.0,0.0,0.0,0.0,115987185.0
CNT_CREDIT_PROLONG,1716428.0,NaN,NaN,NaN,0.00641,0.096224,0.0,0.0,0.0,0.0,9.0


previous_application summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_PREV,1670214.0,NaN,NaN,NaN,1923089.135331,532597.958696,1000001.0,1461857.25,1923110.5,2384279.75,2845382.0
SK_ID_CURR,1670214.0,NaN,NaN,NaN,278357.174099,102814.823849,100001.0,189329.0,278714.5,367514.0,456255.0
NAME_CONTRACT_TYPE,1670214,4,Cash loans,747553,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMT_ANNUITY,1297979.0,NaN,NaN,NaN,15955.120659,14782.137335,0.0,6321.78,11250.0,20658.42,418058.145
AMT_APPLICATION,1670214.0,NaN,NaN,NaN,175233.86036,292779.762386,0.0,18720.0,71046.0,180360.0,6905160.0
AMT_CREDIT,1670213.0,NaN,NaN,NaN,196114.021218,318574.616547,0.0,24160.5,80541.0,216418.5,6905160.0
AMT_DOWN_PAYMENT,774370.0,NaN,NaN,NaN,6697.402139,20921.49541,-0.9,0.0,1638.0,7740.0,3060045.0
AMT_GOODS_PRICE,1284699.0,NaN,NaN,NaN,227847.279283,315396.557937,0.0,50841.0,112320.0,234000.0,6905160.0
WEEKDAY_APPR_PROCESS_START,1670214,7,TUESDAY,255118,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HOUR_APPR_PROCESS_START,1670214.0,NaN,NaN,NaN,12.484182,3.334028,0.0,10.0,12.0,15.0,23.0


installments_payments summary:


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,13605401.0,1.903365e+06,536202.905546,1000001.0,1434191.000,1896520.000,2369094.000,2843499.000
SK_ID_CURR,13605401.0,2.784449e+05,102718.310411,100001.0,189639.000,278685.000,367530.000,456255.000
NUM_INSTALMENT_VERSION,13605401.0,8.566373e-01,1.035216,0.0,0.000,1.000,1.000,178.000
NUM_INSTALMENT_NUMBER,13605401.0,1.887090e+01,26.664067,1.0,4.000,8.000,19.000,277.000
DAYS_INSTALMENT,13605401.0,-1.042270e+03,800.946284,-2922.0,-1654.000,-818.000,-361.000,-1.000
DAYS_ENTRY_PAYMENT,13602496.0,-1.051114e+03,800.585883,-4921.0,-1662.000,-827.000,-370.000,-1.000
AMT_INSTALMENT,13605401.0,1.705091e+04,50570.254429,0.0,4226.085,8884.080,16710.210,3771487.845
AMT_PAYMENT,13602496.0,1.723822e+04,54735.783981,0.0,3398.265,8125.515,16108.425,3771487.845


credit_card_balance summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SK_ID_PREV,3840312.0,NaN,NaN,NaN,1904503.5899,536469.470563,1000018.0,1434385.0,1897122.0,2369327.75,2843496.0
SK_ID_CURR,3840312.0,NaN,NaN,NaN,278324.207289,102704.475133,100006.0,189517.0,278396.0,367580.0,456250.0
MONTHS_BALANCE,3840312.0,NaN,NaN,NaN,-34.521921,26.667751,-96.0,-55.0,-28.0,-11.0,-1.0
AMT_BALANCE,3840312.0,NaN,NaN,NaN,58300.155262,106307.031024,-420250.185,0.0,0.0,89046.68625,1505902.185
AMT_CREDIT_LIMIT_ACTUAL,3840312.0,NaN,NaN,NaN,153807.9574,165145.699525,0.0,45000.0,112500.0,180000.0,1350000.0
AMT_DRAWINGS_ATM_CURRENT,3090496.0,NaN,NaN,NaN,5961.324822,28225.688578,-6827.31,0.0,0.0,0.0,2115000.0
AMT_DRAWINGS_CURRENT,3840312.0,NaN,NaN,NaN,7433.388179,33846.077333,-6211.62,0.0,0.0,0.0,2287098.315
AMT_DRAWINGS_OTHER_CURRENT,3090496.0,NaN,NaN,NaN,288.169582,8201.989345,0.0,0.0,0.0,0.0,1529847.0
AMT_DRAWINGS_POS_CURRENT,3090496.0,NaN,NaN,NaN,2968.804848,20796.887047,0.0,0.0,0.0,0.0,2239274.16
AMT_INST_MIN_REGULARITY,3535076.0,NaN,NaN,NaN,3540.204129,5600.154122,0.0,0.0,0.0,6633.91125,202882.005


In [10]:
# Creating a summary table of the datasets
datasets = {
    "application_train": application_train,
    "bureau": bureau,
    "previous_application": previous_application,
    "installments_payments": installments_payments,
    "credit_card_balance": credit_card_balance,
}

summary_table = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "missing_pct": df.isna().mean().mean() * 100,
        }
        for name, df in datasets.items()
    ]
).assign(missing_pct=lambda frame: frame["missing_pct"].round(2))

display(summary_table)

,dataset,rows,columns,missing_pct
0,application_train,307511,122,24.40
1,bureau,1716428,17,13.50
2,previous_application,1670214,37,17.98
3,installments_payments,13605401,8,0.01
4,credit_card_balance,3840312,23,6.65


In [14]:
#  Loading column descriptions and create a lookup helper
description_path = data_path("HomeCredit_columns_description.csv")

if description_path.exists():
    columns_description = pd.read_csv(description_path, encoding="latin1")
    print_load_check("HomeCredit_columns_description", columns_description)
else:
    columns_description = pd.DataFrame(columns=["Table", "Row", "Description"])
    print(f"Column descriptions file not found: {description_path}")
    print("Add HomeCredit_columns_description.csv to DATA_DIR before using describe_column().")


def describe_column(column_name: str) -> None:
    """Print the Home Credit data dictionary description for a column."""
    matches = columns_description[
        columns_description["Row"].astype(str).str.casefold() == column_name.casefold()
    ]

    if matches.empty:
        print(f"No description found for {column_name!r}.")
        return

    display(matches[["Table", "Row", "Description"]].drop_duplicates())


if not columns_description.empty:
    describe_column("TARGET")

Column descriptions file not found: C:\Users\Kim\Desktop\Loan Default Predictions\HomeCredit_columns_description.csv
Add HomeCredit_columns_description.csv to DATA_DIR before using describe_column().


In [16]:
# Interpretation of the main training file and target variable
applicant_count = len(application_train)
default_rate = application_train["TARGET"].mean() * 100

display(
    Markdown(
        f"The main training file contains {applicant_count:,} applicants with an "
        f"{default_rate:.0f}% default rate. Multiple auxiliary files provide "
        "historical financial behaviour at the individual level."
    )
)

The main training file contains 307,511 applicants with an 8% default rate. Multiple auxiliary files provide historical financial behaviour at the individual level.